# Product Data Preparation

This notebook merges product JSONL files, generates variant-level IDs,
and exports a corpus to `products.csv`.

`doc_id` format: `<parent_asin>-<RANDOM10>` where:
- `parent_asin` is used as the prefix (or `no_parent` if missing)
- `RANDOM10` is a random 10-character alphanumeric suffix.

In [ ]:
import os
import json
import random
import string
import pandas as pd
import numpy as np

DATA_DIR = "data"
OUTPUT_CSV = "data/products.csv"
OUTPUT_METADATA = "data/products_metadata.json"

CATEGORY_FILES = {
    "All Beauty": "meta_All_Beauty.jsonl",
    "Appliances": "meta_Appliances.jsonl",
    "Cell Phones & Accessories": "meta_Cell_Phones_and_Accessories.jsonl",
    "Office Products": "meta_Office_Products.jsonl",
    "Sports & Outdoors": "meta_Sports_and_Outdoors.jsonl",
}

# Reproducible random suffixes for doc_id generation
RANDOM_SEED = 42
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)

# Set to None for full data; use an int only for quick tests
sample_size = 2000

In [7]:
def to_text(x):
    if isinstance(x, list):
        return " ".join(map(str, x))
    if pd.isna(x):
        return ""
    return str(x)


def random_suffix(n=10):
    alphabet = string.ascii_letters + string.digits
    return "".join(random.choices(alphabet, k=n))


def load_jsonl_file(input_file, sample_size=None):
    data = []
    with open(input_file, "rt", encoding="utf-8") as f:
        for i, line in enumerate(f):
            if sample_size is not None and i >= sample_size:
                break
            data.append(json.loads(line))
    return pd.DataFrame(data)

In [8]:
frames = []
for category, file_name in CATEGORY_FILES.items():
    path = os.path.join(DATA_DIR, file_name)
    part = load_jsonl_file(path, sample_size=sample_size)
    part["source_category_file"] = category
    frames.append(part)

df = pd.concat(frames, ignore_index=True)
print("Merged rows:", len(df))
print("Columns:", len(df.columns))
df.head(3)

Merged rows: 10000
Columns: 17


,main_category,title,average_rating,rating_number,features,description,price,images,videos,store,categories,details,parent_asin,bought_together,source_category_file,subtitle,author
0,All Beauty,"Howard LC0008 Leather Conditioner, 8-Ounce (4-...",4.8,10,[],[],NaN,[{'thumb': 'https://m.media-amazon.com/images/...,[],Howard Products,[],{'Package Dimensions': '7.1 x 5.5 x 3 inches; ...,B01CUPMQZE,None,All Beauty,NaN,NaN
1,All Beauty,Yes to Tomatoes Detoxifying Charcoal Cleanser ...,4.5,3,[],[],NaN,[{'thumb': 'https://m.media-amazon.com/images/...,[],Yes To,[],"{'Item Form': 'Powder', 'Skin Type': 'Acne Pro...",B076WQZGPM,None,All Beauty,NaN,NaN
2,All Beauty,Eye Patch Black Adult with Tie Band (6 Per Pack),4.4,26,[],[],NaN,[{'thumb': 'https://m.media-amazon.com/images/...,[],Levine Health Products,[],{'Manufacturer': 'Levine Health Products'},B000B658RI,None,All Beauty,NaN,NaN


In [9]:
# Basic cleanup for ID generation
for c in ["title", "description", "features", "main_category", "store", "parent_asin"]:
    if c in df.columns:
        fill = "" if c != "main_category" else "Unknown"
        df[c] = df[c].fillna(fill)

parent_prefix = df["parent_asin"].astype(str).str.strip()
parent_prefix = np.where(parent_prefix == "", "no_parent", parent_prefix)

# variant-level doc_id: parent_asin + random 10-char suffix
df["doc_id"] = [f"{parent_prefix[i]}-{random_suffix(10)}" for i in range(len(df))]

# collision guard (extremely unlikely, but safe)
dup_no = df.groupby("doc_id").cumcount()
df["doc_id"] = np.where(dup_no == 0, df["doc_id"], df["doc_id"] + "__dup" + dup_no.astype(str))

# human-friendly numeric id for annotation sheets
df = df.reset_index(drop=True)
df["doc_num"] = np.arange(1, len(df) + 1)

print("Unique doc_id:", df["doc_id"].nunique())
df[["doc_num", "doc_id", "parent_asin", "title"]].head(5)

Unique doc_id: 10000


,doc_num,doc_id,parent_asin,title
0,1,B01CUPMQZE-NbrnTP3fAb,B01CUPMQZE,"Howard LC0008 Leather Conditioner, 8-Ounce (4-..."
1,2,B076WQZGPM-nFbmOHnKYa,B076WQZGPM,Yes to Tomatoes Detoxifying Charcoal Cleanser ...
2,3,B000B658RI-XRvj7uff0L,B000B658RI,Eye Patch Black Adult with Tie Band (6 Per Pack)
3,4,B088FKY3VD-YTH8xIZM1J,B088FKY3VD,"Tattoo Eyebrow Stickers, Waterproof Eyebrow, 4..."
4,5,B07NGFDN6G-RcoreogrNw,B07NGFDN6G,Precision Plunger Bars for Cartridge Grips – 9...


In [10]:
# Save frozen corpus
export_cols_front = ["doc_num", "doc_id", "parent_asin", "source_category_file"]
remaining_cols = [c for c in df.columns if c not in export_cols_front]
final_df = df[export_cols_front + remaining_cols]

final_df.to_csv(OUTPUT_CSV, index=False)

metadata = {
    "rows": int(len(final_df)),
    "columns": int(len(final_df.columns)),
    "doc_id_unique": int(final_df["doc_id"].nunique()),
    "sample_size_per_category": sample_size,
    "random_seed": RANDOM_SEED,
    "output_csv": OUTPUT_CSV,
}

with open(OUTPUT_METADATA, "w", encoding="utf-8") as f:
    json.dump(metadata, f, indent=2)

print("Saved:", OUTPUT_CSV)
print("Saved:", OUTPUT_METADATA)
metadata

Saved: data/products.csv
Saved: data/products_metadata.json


{'rows': 10000,
 'columns': 19,
 'doc_id_unique': 10000,
 'sample_size_per_category': 2000,
 'random_seed': 42,
 'output_csv': 'data/products.csv'}